In [12]:
%%time
%%capture

%pip install --user transformers Penman

CPU times: user 56.4 ms, sys: 40.3 ms, total: 96.7 ms
Wall time: 27 s


In [9]:
import penman
import torch
from transformers import AutoModelForSeq2SeqLM, BartTokenizer

In [5]:
%%time
model_name = "xfbai/AMRBART-large-finetuned-AMR3.0-AMRParsing-v2"

# Load tokenizer + model
tokenizer = BartTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.64G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/1.64G [00:00<?, ?B/s]

CPU times: user 5.04 s, sys: 4.09 s, total: 9.13 s
Wall time: 1min 13s


BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(53228, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(53228, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [6]:
sentence = "The boy wants to go to school."

In [7]:
%%time
# Tokenize
inputs = tokenizer(sentence, return_tensors="pt", truncation=True, max_length=512).to(
    device
)

# Generate AMR
with torch.no_grad():
    outputs = model.generate(**inputs, max_length=512, num_beams=5)

# Decode
amr = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Input Sentence:")
print(sentence)
print("\nPredicted AMR:")
print(amr)

Input Sentence:
The boy wants to go to school.

Predicted AMR:
 ( <pointer:0> want-01 :ARG0 ( <pointer:1> boy ) :ARG1 ( <pointer:2> go-01 :ARG1 <pointer:1> :ARG4 ( <pointer:3> school ) ) )</AMR> ( <pointer:4> want-01 :ARG0 <pointer:1> :ARG1 ( <pointer:5> go-02 :ARG0 <pointer:1> :ARG4 <pointer:3> ) ) :ARG2 <pointer:3> )</AMR></AMR> )</AMR>-of ( <pointer:6> cause-01 :ARG0 ( <pointer:7> amr-unknown ) :ARG1 <pointer:4> )</AMR> :ARG1-of ( <pointer:8> cause-01 :ARG0 <pointer:7> ) ) :time ( <pointer:9> date-entity :weekday ( <pointer:10> monday ) ) :mod ( <pointer:11> also ) ) :ARG0-of ( <pointer:12> cause-01 :ARG1 ( <pointer:13> want-01 :ARG1 ( <pointer:14> go-02 :ARG0 ( <pointer:15> boy ) :ARG4 ( <pointer:16> high-school ) ) :polarity ( <pointer:17> amr-unknown ) ) :location ( <pointer:18> there ) ) :ARG1-of <pointer:8> )</AMR> :ARG2-of ( <pointer:19> have-degree-91 :ARG1 <pointer:1> :ARG3 ( <pointer:20> more ) :ARG4 <pointer:16> ) ) :ARG3 ( <pointer:21> more ) ) :op2 ( <pointer:22> more )<

In [11]:
# raw_amr = tokenizer.decode(outputs[0])

# Convert to PENMAN graph
graph = penman.decode(amr)
print("Predicted AMR (PENMAN):")
print(graph)

Missing target:  ( <pointer:0> want-01 :ARG0 ( <pointer:1> boy ) :ARG1 ( <pointer:2> go-01 :ARG1 <pointer:1> :ARG4 ( <pointer:3> school ) ) )</AMR> ( <pointer:4> want-01 :ARG0 <pointer:1> :ARG1 ( <pointer:5> go-02 :ARG0 <pointer:1> :ARG4 <pointer:3> ) ) :ARG2 <pointer:3> )</AMR></AMR> )</AMR>-of ( <pointer:6> cause-01 :ARG0 ( <pointer:7> amr-unknown ) :ARG1 <pointer:4> )</AMR> :ARG1-of ( <pointer:8> cause-01 :ARG0 <pointer:7> ) ) :time ( <pointer:9> date-entity :weekday ( <pointer:10> monday ) ) :mod ( <pointer:11> also ) ) :ARG0-of ( <pointer:12> cause-01 :ARG1 ( <pointer:13> want-01 :ARG1 ( <pointer:14> go-02 :ARG0 ( <pointer:15> boy ) :ARG4 ( <pointer:16> high-school ) ) :polarity ( <pointer:17> amr-unknown ) ) :location ( <pointer:18> there ) ) :ARG1-of <pointer:8> )</AMR> :ARG2-of ( <pointer:19> have-degree-91 :ARG1 <pointer:1> :ARG3 ( <pointer:20> more ) :ARG4 <pointer:16> ) ) :ARG3 ( <pointer:21> more ) ) :op2 ( <pointer:22> more )</AMR> :time ( <pointer:23> now )</AMR> :ARG0 ( 

Predicted AMR (PENMAN):
Graph(
  [('<pointer', ':instance', None),
   ('<pointer', ':0>', 'want-01'),
   ('<pointer', ':ARG0', '<pointer'),
   ('<pointer', ':instance', None),
   ('<pointer', ':1>', 'boy'),
   ('<pointer', ':ARG1', '<pointer'),
   ('<pointer', ':instance', None),
   ('<pointer', ':2>', 'go-01'),
   ('<pointer', ':ARG1', '<pointer'),
   ('<pointer', ':1>', None),
   ('<pointer', ':ARG4', '<pointer'),
   ('<pointer', ':instance', None),
   ('<pointer', ':3>', 'school')],
  epidata={('<pointer', ':0>', 'want-01'): [],
    ('<pointer', ':ARG0', '<pointer'): [Push(<pointer)],
    ('<pointer', ':1>', 'boy'): [],
    ('<pointer', ':instance', None): [POP],
    ('<pointer', ':ARG1', '<pointer'): [Push(<pointer)],
    ('<pointer', ':2>', 'go-01'): [],
    ('<pointer', ':1>', None): [],
    ('<pointer', ':ARG4', '<pointer'): [Push(<pointer)],
    ('<pointer', ':3>', 'school'): []})
